In [1]:
import random
import numpy as np
import pandas as pd
import seaborn as sns
import os
import sys
import matplotlib.pyplot as plt
from numpy.linalg import *

import datetime

import warnings
import glob
import io
from google.colab import drive
warnings.filterwarnings("ignore")

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
#from sklearn.linear_model import Ridge #

from sklearn.metrics import r2_score

In [2]:
train_df = pd.read_csv('/kaggle/input/next-day-air-temperature-forecast-challenge/train_dataset.csv')

In [3]:
station_df = pd.read_csv('/kaggle/input/next-day-air-temperature-forecast-challenge/station_info.csv')

In [4]:
station_df.drop(index=8, inplace=True)

In [5]:
feature_prefixes = ['cloud_cover_', 'dew_point_', 'humidity_',
            'local_pressure_', 'min_cloud_height_', 'precipitation_',
            'sea_level_pressure_', 'snow_depth_', 'sunshine_duration_',
            'surface_temp_', 'vapor_pressure_', 'visibility_', 'wind_speed_',
            'wind_direction_']


In [6]:
features_n = ['id', 'station', 'station_name', 'date']
for feature in feature_prefixes:
  for i in range(24):
    features_n.append(feature+str(i))
features_n.append('climatology_temp')
features_n.append('target')

train_df = train_df[features_n]  

In [7]:
# 드롭할 접두어 리스트
drop_prefixes = [
    # 'cloud_cover_'
    # , 'sunshine_duration_'
    # , 'min_cloud_height_'
    # , 'precipitation_'
    # , 'snow_depth_'
    # , 'wind_direction_'
]

In [8]:
mean_features = [
    'surface_temp_',
    'dew_point_',
    'humidity_',
    'local_pressure_',
    'sea_level_pressure_',
    'vapor_pressure_', 
    'min_cloud_height_' 
    #, 'wind_speed_'#
    #, 'sunshine_duration_'#
    , 'cloud_cover_'
]
max_features = [
    'wind_speed_'
    , 'cloud_cover_'
    , 'snow_depth_'
    #'precipitation_'
    ,'sea_level_pressure_'
    #,'surface_temp_'
]
sum_features = [
    'precipitation_', 'sunshine_duration_',
    #'vapor_pressure_',
    # 'surface_temp_'
]
min_features = [
    # 'dew_point_',
    # 'vapor_pressure_',
    'visibility_'
    ,'sea_level_pressure_'
    #,'surface_temp_'
]
std_features = [
    # 'humidity_',
    #'wind_speed_',
    # 'wind_direction_'
]

In [9]:
# IQR 기준 이상치 확인하는 함수
def changeOutliers(tdf, features, method='linear'):
    for prefix in features:
        for i in range(24):
            col = f"{prefix}{i}"
            if col in tdf.columns:
                # 이상치 탐지 (IQR)
                q1 = tdf[col].quantile(0.25)
                q3 = tdf[col].quantile(0.75)
                iqr = q3 - q1
                Min = q1 - 1.5 * iqr
                Max = q3 + 1.5 * iqr

                # 이상치 및 기존 결측치 -> NaN 처리
                tdf[col] = tdf[col].where((tdf[col] >= Min) & (tdf[col] <= Max), np.nan)

                # NaN 보간 (이상치 + 기존 결측치 포함)
                #tdf[col] = tdf[col].interpolate(method=method)

                # 아직 남은 NaN -> 평균 대체
                #tdf[col] = tdf[col].fillna(tdf[col].mean())
    return tdf

In [10]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'fall'

In [11]:
# 요약 통계 변수 생성 함수
def summarize_time_variables(tdf, features):
    new_df = tdf[['id', 'station', 'station_name', 'date']].copy()

    for var in mean_features:
        cols = [f'{var}{i}' for i in range(24) if f'{var}{i}' in tdf.columns]
        new_df[f'{var}mean'] = tdf[cols].mean(axis=1)

    for var in max_features:
        cols = [f'{var}{i}' for i in range(24) if f'{var}{i}' in tdf.columns]
        new_df[f'{var}max'] = tdf[cols].max(axis=1)

    for var in sum_features:
        cols = [f'{var}{i}' for i in range(24) if f'{var}{i}' in tdf.columns]
        new_df[f'{var}sum'] = tdf[cols].sum(axis=1)

    for var in min_features:
        cols = [f'{var}{i}' for i in range(24) if f'{var}{i}' in tdf.columns]
        new_df[f'{var}min'] = tdf[cols].min(axis=1)

    #wind_direction circular mean 변환
    #wind = [f'wind_direction_{i}' for i in range(24)] #


    
    
    # for var in std_features:
    #     cols = [f'{var}{i}' for i in range(24) if f'{var}{i}' in tdf.columns]
    #     new_df[f'{var}std'] = tdf[cols].std(axis=1)


    # for var in features:
    #     cols = [f"{var}{i}" for i in range(24) if f"{var}{i}" in tdf.columns]
    #     if cols:
    #         new_df[f'{var}mean'] = tdf[cols].mean(axis=1)
    #         new_df[f'{var}max'] = tdf[cols].max(axis=1)
    #         new_df[f'{var}min'] = tdf[cols].min(axis=1)
    #         new_df[f'{var}std'] = tdf[cols].std(axis=1)
    #         new_df[f'{var}sum'] = tdf[cols].sum(axis=1)
    #         new_df[f'{var}change'] = new_df[f'{var}max'] - new_df[f'{var}min']
    if 'climatology_temp' in tdf.columns:
        new_df['climatology_temp'] = tdf['climatology_temp']
    if 'target' in tdf.columns:
        new_df['target'] = tdf['target']


    new_df['month'] = pd.to_datetime(new_df['date'], format='%m-%d', errors='coerce').dt.month

    new_df['season'] = new_df['month'].apply(get_season)
    new_df = pd.get_dummies(new_df, columns=['season'])

    
    return new_df

In [12]:
#def add_time_band_features(df):
    # 6구간 시간대 평균 파생
#    df['early_morning_temp_mean'] = df[[f'surface_temp{i}' for i in range(0, 6)]].mean(axis=1)
#    df['morning_temp_mean']       = df[[f'surface_temp{i}' for i in range(6, 10)]].mean(axis=1)
#    df['late_morning_temp_mean']  = df[[f'surface_temp{i}' for i in range(10, 12)]].mean(axis=1)
#    df['afternoon_temp_mean']     = df[[f'surface_temp{i}' for i in range(12, 18)]].mean(axis=1)
#    df['evening_temp_mean']       = df[[f'surface_temp{i}' for i in range(18, 21)]].mean(axis=1)
#    df['night_temp_mean']         = df[[f'surface_temp{i}' for i in range(21, 24)]].mean(axis=1)

    # 6구간 평균 요약
#    time_bands = ['early_morning_temp_mean', 'morning_temp_mean',
#                  'late_morning_temp_mean', 'afternoon_temp_mean',
#                  'evening_temp_mean', 'night_temp_mean']
#    df['time_band_avg'] = df[time_bands].mean(axis=1)
#    df['time_band_range'] = df[time_bands].max(axis=1) - df[time_bands].min(axis=1)

#    return df


In [13]:
# 추가 파생 feature
def add_custom_features(df):
    
    # 이슬점 * 습도: 포화도 및 공기 중 수분량의 간접 지표
    df['dew_humidity_interaction'] = df['dew_point_mean'] * df['humidity_mean'] # 1
    
    # 온도 대비 기압 비율
    df['temp_pressure_ratio'] = df['surface_temp_mean'] / (df['local_pressure_mean'] + 1e-5) # 2

    df['humidity_precip_interaction'] = df['humidity_mean'] * df['precipitation_sum'] #
    #df['wind_speed_max_squared'] = df['wind_speed_max'] ** 2 #
    df['pressure_temp_ratio'] = df['sea_level_pressure_mean'] / df['climatology_temp'].replace(0, np.nan)

    df['pressure_range'] = df['sea_level_pressure_max'] - df['sea_level_pressure_min'] ##

    df['humidity_pressure_ratio'] = df['humidity_mean'] / (df['sea_level_pressure_mean'] + 1e-5)
    
    return df

In [14]:
## 전처리 - 결측치 처리
def preprocessing(tdf, features, drop_features):
    # 결측치 통일
    tdf.replace([-9999, None, ''], np.nan, inplace=True)

    # 이상치 + 결측치 → NaN → 보간 → 평균
    # tdf = changeOutliers(tdf, feature_prefixes)
    
    # 드롭할 피처 접두어 목록이 주어졌다면 해당 열 제거
    if drop_prefixes:
        drop_cols = [f"{prefix}{i}" for prefix in drop_prefixes for i in range(24)]
        drop_cols = [col for col in drop_cols if col in tdf.columns]
        tdf.drop(columns=drop_cols, inplace=True)

    # 남은 NaN → 0으로 최종 대체
    #tdf.fillna(0, inplace=True)

    return tdf

In [15]:
#관측소 지역명
station = [
    '동두천'
    ,'파주'
    ,'강화'
    ,'인천'
    ,'이천'
    ,'양평'
    ,'서울'
    ,'수원'
]
#관측소 특성 추가 features
station_features = [
    '위도'
    ,'경도'
    ,'노장해발고도(m)'
    # ,'기압계(관측장비지상높이(m))'
    # ,'기온계(관측장비지상높이(m))'
    # ,'풍속계(관측장비지상높이(m))'
    # ,'강우계(관측장비지상높이(m))'
]

In [16]:
train_df_cleaned = preprocessing(train_df.copy(), feature_prefixes, drop_prefixes)

train_df_final = summarize_time_variables(train_df_cleaned, feature_prefixes)
train_df_final = add_custom_features(train_df_final) 

X_train = train_df_final.drop(columns=['target','station', 'station_name', 'date', 'id'])

y_train = train_df_final['target']

feature_names = X_train.columns.tolist()

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)

In [17]:
X_train

,surface_temp_mean,dew_point_mean,humidity_mean,local_pressure_mean,sea_level_pressure_mean,vapor_pressure_mean,min_cloud_height_mean,cloud_cover_mean,wind_speed_max,cloud_cover_max,...,season_fall,season_spring,season_summer,season_winter,dew_humidity_interaction,temp_pressure_ratio,humidity_precip_interaction,pressure_temp_ratio,pressure_range,humidity_pressure_ratio
0,-5.583333,-16.808333,46.875000,1019.225000,1034.412500,1.666667,38.444444,1.791667,4.1,9.0,...,False,False,False,True,-787.890625,-0.005478,0.000000,-382.104881,5.0,0.045316
1,-5.762500,-17.466667,45.500000,1019.962500,1035.141667,1.570833,NaN,0.000000,3.6,0.0,...,False,False,False,True,-794.733333,-0.005650,0.000000,-283.878224,4.1,0.043955
2,-5.320833,-17.645833,43.583333,1020.916667,1036.045833,1.554167,NaN,0.000000,1.5,0.0,...,False,False,False,True,-769.064236,-0.005212,0.000000,-384.483543,3.6,0.042067
3,-3.920833,-11.966667,53.166667,1016.854167,1031.783333,2.591667,28.272727,1.208333,2.8,4.0,...,False,False,False,True,-636.227778,-0.003856,0.000000,-412.418749,8.7,0.051529
4,-3.391667,-15.729167,40.583333,1013.066667,1027.925000,1.958333,32.000000,0.250000,3.3,5.0,...,False,False,False,True,-638.342014,-0.003348,0.000000,-391.590476,3.3,0.039481
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10500,16.954167,11.420833,84.250000,1005.504167,1011.129167,13.508333,8.250000,4.916667,2.2,9.0,...,True,False,False,False,962.205208,0.016861,0.000000,76.311635,2.8,0.083323
10501,16.583333,8.620833,73.375000,1009.433333,1015.062500,11.366667,6.800000,1.250000,1.8,6.0,...,True,False,False,False,632.553646,0.016428,0.000000,87.276984,4.9,0.072286
10502,15.104167,6.179167,75.833333,1014.162500,1019.879167,9.525000,NaN,1.625000,1.9,7.0,...,True,False,False,False,468.586806,0.014893,0.000000,83.474471,3.7,0.074355
10503,15.604167,8.370833,79.541667,1014.841667,1020.537500,11.075000,13.900000,2.208333,2.6,8.0,...,True,False,False,False,665.830035,0.015376,0.000000,77.956759,5.3,0.077941


In [18]:
#print(train_df.columns.tolist())

In [19]:
M = 20                           # 앙상블할 모델 수
feature_ratio = 0.6                # 각 모델에 사용할 특성 비율 
models, feature_sets = [], []

# 2) 모델 학습
for seed in range(M):
    # 랜덤 시드 고정 후 특성 샘플링
    np.random.seed(seed)
    features = np.random.choice(feature_names,
                                size=int(len(feature_names)*feature_ratio),
                                replace=False).tolist()
    feature_sets.append(features)
    
    X_tr_sub = X_train[features]
    X_val_sub = X_val[features]
    
    # XGBoost 모델
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=9,
        subsample=0.8,
        colsample_bytree=0.8,
        alpha=1.0,
        reg_lambda=1.0,
        random_state=seed,      
    )
    
    model.fit(
        X_tr_sub, y_train,
        eval_set=[(X_val_sub, y_val)],
        early_stopping_rounds=50,
        verbose=False
    )
    models.append(model)

#예측값 데이터프레임 생성
preds = {}
for i in range(M):
    preds[f'y_pred_{i}'] = models[i].predict(X_val[feature_sets[i]])
    
train_pred_df = pd.DataFrame(preds)

#앙상블 예측 - 선형회귀
lr_model = LinearRegression()
lr_model.fit(train_pred_df, y_val)

ensemble_pred = lr_model.predict(train_pred_df)

# 클리핑 적용 (-30도 ~ 50도 예시)
ensemble_pred = np.clip(ensemble_pred, -20, 40)

rmse = mean_squared_error(y_val, ensemble_pred, squared=False)
print(f"Ensemble RMSE: {rmse:.4f}")

r2 = r2_score(y_val, ensemble_pred)
print(f"Ensemble R^2 Score: {r2:.4f}")

#=================================================================
# 앙상블 예측 - Ridge 회귀
#ridge_model = Ridge(alpha=1.0)  # 규제 강도는 0.1, 1.0, 10.0 실험 가능
#ridge_model.fit(train_pred_df, y_val)

#ensemble_pred = ridge_model.predict(train_pred_df)

# 성능 평가
#rmse = mean_squared_error(y_val, ensemble_pred, squared=False)
#print(f"Ridge Ensemble RMSE: {rmse:.4f}")

#r2 = r2_score(y_val, ensemble_pred)
#print(f"Ridge Ensemble R^2 Score: {r2:.4f}")

Ensemble RMSE: 1.2148
Ensemble R^2 Score: 0.8192


In [20]:
print("Validation Prediction Min:", ensemble_pred.min())
print("Validation Prediction Max:", ensemble_pred.max())

Validation Prediction Min: -12.3316145
Validation Prediction Max: 9.238281


In [21]:
# 2단계 메타 모델을 XGBoost로 변경
#meta_model = XGBRegressor(
#    n_estimators=500,
#    learning_rate=0.05,
#    max_depth=1,##
#    random_state=42
#)

#meta_model.fit(train_pred_df, y_val)

# 최종 예측
#ensemble_pred = meta_model.predict(train_pred_df)

# 평가
#rmse = mean_squared_error(y_val, ensemble_pred, squared=False)
#print(f"Ensemble (2nd stage XGBoost) RMSE: {rmse:.4f}")

In [22]:
# 모든 모델의 중요도를 평균 집계
importance_records = []

for model, features in zip(models, feature_sets):
    importance_records.append(pd.Series(model.feature_importances_, index=features))

importance_df = pd.concat(importance_records, axis=1).fillna(0).mean(axis=1).sort_values(ascending=False)

# 출력
print(importance_df.head(20))

season_winter                  0.244236
season_summer                  0.057657
dew_point_mean                 0.052323
season_fall                    0.045276
climatology_temp               0.041911
dew_humidity_interaction       0.041598
vapor_pressure_mean            0.039893
month                          0.039553
precipitation_sum              0.037218
snow_depth_max                 0.033887
pressure_temp_ratio            0.032311
surface_temp_mean              0.031390
humidity_precip_interaction    0.027282
humidity_mean                  0.026057
sea_level_pressure_mean        0.025127
temp_pressure_ratio            0.024480
season_spring                  0.024427
humidity_pressure_ratio        0.023367
sea_level_pressure_max         0.021322
sea_level_pressure_min         0.019155
dtype: float32


In [23]:
# model = XGBRegressor(
#     n_estimators=1000,
#     max_depth=8,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     alpha=5.0,
#     reg_lambda=5.0,
#     gamma=0.1,
#     random_state=42
# )
# model.fit(
#     X_train, y_train,
#     eval_set=[(X_val, y_val)],
#     eval_metric='rmse',
#     early_stopping_rounds=20,
#     verbose=50
# )

# y_pred = model.predict(X_val)
# rmse = mean_squared_error(y_val, y_pred, squared=False)
# print(f"Validation RMSE: {rmse:.4f}")

In [24]:
test_df = pd.read_csv('/kaggle/input/test-dataset/test_dataset (1).csv')

In [25]:
test_df_cleaned = preprocessing(test_df.copy(), feature_prefixes, drop_prefixes)

test_df_final = summarize_time_variables(test_df_cleaned, feature_prefixes)
test_df_final = add_custom_features(test_df_final)

X_test_final = test_df_final.drop(columns=['station', 'station_name', 'date', 'id'])


X_test_final = X_test_final[X_train.columns]

test_pred = {}
for i in range(M):
    test_pred[f'y_pred_{i}'] = models[i].predict(X_test_final[feature_sets[i]])

test_pred_df = pd.DataFrame(test_pred)

#클리핑
test_df_final['target'] = np.clip(lr_model.predict(test_pred_df), -20, 40)

test_df_final['target'] = lr_model.predict(test_pred_df)

# test_df_final['target'] = model.predict(X_test_final)

#test_df_final['target'] = ridge_model.predict(test_pred_df)

In [26]:
# 1. 테스트 데이터 전처리
#test_df_cleaned = preprocessing(test_df.copy(), feature_prefixes, drop_prefixes)
#test_df_final = summarize_time_variables(test_df_cleaned, feature_prefixes)
#test_df_final = add_custom_features(test_df_final)

# 2. 학습 시 사용한 컬럼 순서 맞추기
#X_test_final = test_df_final.drop(columns=['station', 'station_name', 'date', 'id'])

#X_test_final = X_test_final[X_train.columns]

# 3. 1단계 모델들의 예측값 생성
#test_pred = {}
#for i in range(M):
#    test_pred[f'y_pred_{i}'] = models[i].predict(X_test_final[feature_sets[i]])

#test_pred_df = pd.DataFrame(test_pred)

# 4. 2단계 XGBoost 메타 모델을 사용해 최종 예측
#test_df_final['target'] = meta_model.predict(test_pred_df)

In [27]:
sub_df = pd.read_csv('/kaggle/input/next-day-air-temperature-forecast-challenge/submission_sample.csv')

In [28]:
sub_df = test_df_final[['id', 'target']]

In [29]:
sub_df

,id,target
0,0,2.035796
1,1,2.298392
2,2,0.667885
3,3,1.645841
4,4,0.607561
...,...,...
2999,2999,1.340390
3000,3000,-7.437497
3001,3001,-2.653902
3002,3002,0.710857


In [30]:
sub_df.to_csv('./submission.csv', index=False)